# Kamen Rider Image Convolutions with MPI and CUDA

Parallel image convolution using **MPI** and **CUDA**.

This notebook executes:

- MPI parallel processing
- CUDA GPU acceleration
- OpenCV image processing
- `parallel_image` executable

Project: **Schryzon/mpyCUDA**  
Course: **Parallel Processing A**
By: I Nyoman Widiyasa Jayananda (F1D02410053)

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Jay-IF24-mpyCUDA'
REPO_URL   = 'https://github.com/Schryzon/mpyCUDA.git'

if not os.path.exists(DRIVE_PATH):
    print('Repository not found in Drive. Cloning now...')
    !git clone "{REPO_URL}" "{DRIVE_PATH}"
    print('Clone complete!')
else:
    print('Repository already exists in Drive. Pulling latest changes...')
    !git -C "{DRIVE_PATH}" pull

# Create a fast symlink in /content so all relative paths work correctly
WORK_DIR = '/content/mpyCUDA'
if not os.path.exists(WORK_DIR):
    !ln -s "{DRIVE_PATH}" "{WORK_DIR}"

print(f'Working directory: {WORK_DIR}')

%cd mpyCUDA/Kamen-Rider-Image-Convolution

In [ ]:
!apt-get update -y
!apt-get install -y build-essential
!apt-get install -y openmpi-bin openmpi-common libopenmpi-dev
!apt-get install -y libopencv-dev
!apt-get install -y cmake

In [ ]:
!echo "--- MPI ---" && mpirun --version
!echo "--- nvcc ---" && nvcc --version
!echo "--- OpenCV ---" && pkg-config --modversion opencv4

In [ ]:
!tree -L 3

In [ ]:
SCRIPTS_DIR = f'{WORK_DIR}/Kamen-Rider-Image-Convolution/scripts'

print('Compiling parallel_conv and parallel_image...')
result = !make -C "{SCRIPTS_DIR}" all 2>&1
print('\n'.join(result))

# Verify binaries exist
for binary in ['parallel_conv', 'parallel_image']:
    path = f'{SCRIPTS_DIR}/{binary}'
    if os.path.exists(path):
        print(f'✅ {binary} compiled successfully')
    else:
        print(f'❌ {binary} FAILED to compile — check output above')

In [ ]:
!ls

In [ ]:
from pathlib import Path
import subprocess, re
import numpy as np
import matplotlib.pyplot as plt

procs = [2, 4, 6, 8, 16, 24, 30, 48, 50]

images = [
    "decade.jpg",
    "kuuga.jpg",
    "w_lunatrigger.png",
    "ryuki.jpg"
]

input_prefix = "./images/"
output_prefix = "./images/output"

MPIEXEC = "mpirun"
PARALLEL_EXE = "./scripts/parallel_image"

ops = ["blur", "edge", "sobel", "sharpen", "emboss"]

time_re = re.compile(r"Time taken:\s*([0-9.]+)")

In [ ]:
def run_op(op: str):
    times = []

    for image in images:
        stem = Path(image).stem
        in_img = str(Path(input_prefix) / image)

        out_dir = Path(output_prefix) / op / stem
        out_dir.mkdir(parents=True, exist_ok=True)

        for p in procs:

            out_img = str(out_dir / f"{stem}_{p}.jpg")

            cmd = [
                MPIEXEC,
                "-np", str(p),
                PARALLEL_EXE,
                in_img,
                out_img,
                op,
            ]

            print("Running:", " ".join(cmd))

            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True
            )

            combined = (
                (result.stdout or "")
                + "\n"
                + (result.stderr or "")
            )

            m = time_re.search(combined)

            if not m:
                raise RuntimeError(
                    f"Timing parse failed:\n{combined}"
                )

            t = float(m.group(1))
            times.append(t)

    return times

In [ ]:
times_by_op = {}

for op in ops:
    times_by_op[op] = run_op(op)

times_by_op

In [ ]:
n_images = len(images)
n_procs = len(procs)

for op in ops:

    times_arr = np.array(
        times_by_op[op],
        dtype=float
    ).reshape(n_images, n_procs)

    speedup_arr = times_arr[:, [0]] / times_arr

    # Execution Time Plot
    plt.figure()

    for i, img in enumerate(images):
        plt.plot(
            procs,
            times_arr[i],
            marker="o",
            label=img
        )

    plt.xlabel("Processors")
    plt.ylabel("Execution Time (s)")
    plt.title(f"Execution Time vs Processors ({op})")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Speedup Plot
    plt.figure()

    for i, img in enumerate(images):
        plt.plot(
            procs,
            speedup_arr[i],
            marker="o",
            label=img
        )

    plt.xlabel("Processors")
    plt.ylabel("Speedup")
    plt.title(f"Speedup vs Processors ({op})")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt

P = 50

cwd = Path(os.getcwd())

def _read_img(path: Path):
    img = plt.imread(str(path))

    if (
        isinstance(img, np.ndarray)
        and img.ndim == 3
        and img.shape[2] == 4
    ):
        img = img[:, :, :3]

    return img

n_rows = len(images)
n_cols = 1 + len(ops)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.2 * n_cols, 3.2 * n_rows),
    squeeze=False
)

for r, image in enumerate(images):

    stem = Path(image).stem
    in_path = (cwd / input_prefix / image).resolve()

    ax = axes[r, 0]

    if in_path.exists():
        ax.imshow(_read_img(in_path))
        ax.set_title(f"{image}\nOriginal")
    else:
        ax.text(0.5, 0.5, "Missing")
    
    ax.axis("off")

    for c, op in enumerate(ops, start=1):

        out_path = (
            cwd /
            output_prefix /
            op /
            stem /
            f"{stem}_{P}.jpg"
        ).resolve()

        ax = axes[r, c]

        if out_path.exists():
            ax.imshow(_read_img(out_path))
            ax.set_title(f"{op} (p={P})")
        else:
            ax.text(0.5, 0.5, "Missing")

        ax.axis("off")

plt.tight_layout()
plt.show()